In [0]:
storage_account_key = dbutils.secrets.get(scope="adb-secret-scope", key="gen2key")

storage_account = "stgdataengacctlusprod"
container = "autoloader"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

df = spark.read.csv(
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/source/orders",
    header=True
)

display(df)

In [0]:
display(dbutils.fs.ls("abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/source/orders"))

**Create the Auto Loader Notebook**

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("OrderID", IntegerType()),
    StructField("Customer", StringType()),
    StructField("Product", StringType()),
    StructField("Amount", IntegerType())
])

**Read Using Auto Loader**

In [0]:
df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("cloudFiles.schemaLocation", "abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/schema/orders") #Test Schema Evolution
         .load("abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/source/orders/")
)

# Nothing runs yet because this is a streaming DataFrame.
# Run it again. Auto Loader will detect the new PaymentMode column and evolve the schema automatically.

**Write to Delta**

In [0]:
query = (
    df.writeStream
      .format("delta")
      .option("checkpointLocation",
              "abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/checkpoint/orders")
      .trigger(availableNow=True)
      .start("abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/bronze/orders")
)

**Wait Until Completion**

In [0]:
query.awaitTermination()

**Verify Bronze Data**
_Read the Delta files:_

In [0]:
display(
    spark.read.format("delta")
         .load("abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/bronze/orders")
)

**Check the Checkpoint**

In [0]:
display(dbutils.fs.ls("abfss://autoloader@stgdataengacctlusprod.dfs.core.windows.net/checkpoint/orders"))

In [0]:
df.printSchema()